# Guild or Functional Richness Analysis

Requires a species-to-guild mapping file. Notebook auto-checks for mapping and proceeds if found.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='paper')
RANDOM_SEED = 42


In [ ]:
mapping_candidates = [Path('stateProvince/species_guild_mapping.csv'), Path('species_guild_mapping.csv'), Path('../species_guild_mapping.csv')]
map_path = next((m for m in mapping_candidates if m.exists()), None)
if map_path is None:
    print('No guild mapping file found. Expected columns: verbatimScientificName, guild')
    print('Create species_guild_mapping.csv and rerun this notebook.')
else:
    print('Using mapping:', map_path.resolve())


In [ ]:
if map_path is not None:
    p = next((x for x in [Path('file6.csv'), Path('../file6.csv'), Path('../../file6.csv')] if x.exists()), None)
    cols = ['stateProvince','verbatimScientificName','decimalLatitude','decimalLongitude','eventDate']
    df = pd.read_csv(p, usecols=cols, low_memory=False).dropna(subset=cols)
    for c in ['decimalLatitude','decimalLongitude']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=['decimalLatitude','decimalLongitude'])
    lon0 = float(df['decimalLongitude'].median())
    lat0 = float(df['decimalLatitude'].median())
    lat_rad = np.radians(df['decimalLatitude'].to_numpy())
    df['x_km'] = (df['decimalLongitude'].to_numpy() - lon0) * 111.320 * np.cos(lat_rad)
    df['y_km'] = (df['decimalLatitude'].to_numpy() - lat0) * 110.574
    df['grid_x'] = np.floor(df['x_km']/5.0).astype(int)
    df['grid_y'] = np.floor(df['y_km']/5.0).astype(int)
    df['cell_id'] = df['stateProvince'].astype(str) + '|' + df['grid_x'].astype(str) + '_' + df['grid_y'].astype(str)
    df['_key'] = df['cell_id'] + '|' + df['verbatimScientificName']
    rng = np.random.default_rng(RANDOM_SEED)
    df['_r'] = rng.random(len(df))
    thin = df.sort_values('_r').groupby('_key', as_index=False).head(1)
    mapping = pd.read_csv(map_path)
    mapping['verbatimScientificName'] = mapping['verbatimScientificName'].astype(str).str.strip()
    thin['verbatimScientificName'] = thin['verbatimScientificName'].astype(str).str.strip()
    joined = thin.merge(mapping[['verbatimScientificName','guild']], on='verbatimScientificName', how='inner')
    guild_cell = joined.groupby(['stateProvince','cell_id'], as_index=False).agg(guild_richness=('guild','nunique'))
    guild_dist = guild_cell.groupby('stateProvince', as_index=False).agg(mean_guild_richness=('guild_richness','mean'), n_cells=('cell_id','count')).sort_values('mean_guild_richness', ascending=False)
    display(guild_dist.head(20))
    plt.figure(figsize=(9.5, max(5,0.28*len(guild_dist))))
    plt.barh(guild_dist['stateProvince'], guild_dist['mean_guild_richness'], color='#148f77')
    plt.title('District mean guild richness (thinned)')
    plt.tight_layout()
    plt.show()
